# Logistic Regression Model

This notebook builds the first hallucination detection model using TF-IDF text representation and Logistic Regression.


In [7]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score, GridSearchCV

import joblib

## 1. Load the Processed Dataset

In [8]:
train_data = pd.read_csv("halu_train.csv")
test_data = pd.read_csv("halu_test.csv")

print("Training data:", train_data.shape)
print("Testing data:", test_data.shape)

train_data.head()

Training data: (16000, 5)
Testing data: (4000, 5)


,knowledge,question,answer,label,text
0,Vincenzo Salvatore Carmelo Francesco Bellini (...,"Who has knowledge of more arts, Vincenzo Belli...",Igor Fyodorovich Stravinsky,0,Knowledge: Vincenzo Salvatore Carmelo Francesc...
1,Vincenzo Salvatore Carmelo Francesco Bellini (...,"Who has knowledge of more arts, Vincenzo Belli...",Vincenzo Bellini had more knowledge.,1,Knowledge: Vincenzo Salvatore Carmelo Francesc...
2,"The Naga Chili, closely related to the Bhut jo...","The Naga Chili, closely related to the Bhut jo...",Bangladesh,0,"Knowledge: The Naga Chili, closely related to ..."
3,"The Naga Chili, closely related to the Bhut jo...","The Naga Chili, closely related to the Bhut jo...",The Naga morich is from India.,1,"Knowledge: The Naga Chili, closely related to ..."
4,"The Mega Powers consisted of Hulk Hogan, and ...",What was the cause of the death of this Americ...,drug and alcohol overdose,0,Knowledge: The Mega Powers consisted of Hulk ...


## 2. Prepare Training Data

In [9]:
X_train = train_data["text"]
y_train = train_data["label"]

# Used to keep the two answers of the same question
# in the same cross-validation fold
groups = train_data["question"]

print("Training samples:", len(X_train))
print("Labels:", y_train.value_counts())

Training samples: 16000
Labels: label
0    8000
1    8000
Name: count, dtype: int64


## 3. Build TF-IDF + Logistic Regression Pipeline

In [10]:
logistic_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2)
    )),

    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

logistic_pipeline

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
                ('logreg', LogisticRegression(max_iter=1000, random_state=42))])

## 4. Baseline Cross-Validation

In [11]:
cv = GroupKFold(n_splits=5)

baseline_scores = cross_val_score(
    logistic_pipeline,
    X_train,
    y_train,
    groups=groups,
    cv=cv,
    scoring="f1",
    n_jobs=-1
)

print("F1-score for each fold:", baseline_scores)
print("Mean F1-score:", baseline_scores.mean())

F1-score for each fold: [0.69979578 0.70639731 0.69632626 0.69700103 0.69184086]
Mean F1-score: 0.6982722477114973


## 5. Hyperparameter Tuning

In [12]:
param_grid = {
    "tfidf__max_features": [10000, 20000],
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "logreg__C": [0.1, 1, 10]
}

In [13]:
grid_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(
    X_train,
    y_train,
    groups=groups
)

Fitting 5 folds for each of 12 candidates, totalling 60 fits


GridSearchCV(cv=GroupKFold(n_splits=5, random_state=None, shuffle=False),
             estimator=Pipeline(steps=[('tfidf',
                                        TfidfVectorizer(max_features=20000,
                                                        ngram_range=(1, 2))),
                                       ('logreg',
                                        LogisticRegression(max_iter=1000,
                                                           random_state=42))]),
             n_jobs=-1,
             param_grid={'logreg__C': [0.1, 1, 10],
                         'tfidf__max_features': [10000, 20000],
                         'tfidf__ngram_range': [(1, 1), (1, 2)]},
             scoring='f1', verbose=1)

In [14]:
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1-score:")
print(grid_search.best_score_)

Best Parameters:
{'logreg__C': 10, 'tfidf__max_features': 20000, 'tfidf__ngram_range': (1, 2)}

Best Cross-Validation F1-score:
0.7182060301465286


## 6. Save the Best Logistic Regression Model

In [15]:
best_logistic_model = grid_search.best_estimator_

joblib.dump(
    best_logistic_model,
    "best_logistic_regression_model.pkl"
)

print("Best Logistic Regression model saved successfully.")

Best Logistic Regression model saved successfully.
